In [1]:
%cd ../

/Users/hoangle/Projects/fwo_models


In [2]:
import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer
from tqdm import tqdm

# Load meal names

In [3]:
meal_names = pd.read_excel("data/processed/phase_4/dim_meal_names.xlsx")
meal_names.head()

meal_names.shape

(1060, 2)

In [4]:
pcs = pd.read_excel("data/processed/phase_4/dim_pieces_per_dish.xlsx")
pcs.head()

,date,restaurant,meal_type,pcs,meal_id
0,2023-01-02,che,fish,78,9500055
1,2023-01-02,che,vegan,84,6128
2,2023-01-02,che,meat,165,9500160
3,2023-01-03,che,vegetarian,29,1270
4,2023-01-03,che,fish,105,6156


In [5]:
meal_ids = pcs['meal_id'].unique()

meal_names = meal_names[meal_names['meal_id'].isin(meal_ids)]

meal_names.shape

(470, 2)

# Encode

In [6]:
device = "mps"
tqdm.pandas()


EMBEDDING_MODEL_NAME = "jinaai/jina-embeddings-v3"
tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME)
model     = AutoModel.from_pretrained(EMBEDDING_MODEL_NAME, trust_remote_code=True, from_tf=False, use_flash_attn=False).to(device)


In [8]:
outputs = model.encode(meal_names['meal'].tolist(), task="text-matching")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [9]:
outputs

array([[ 0.01174561, -0.03860296,  0.02718598, ...,  0.02528961,
        -0.01533702,  0.03611615],
       [ 0.01333622, -0.04670437,  0.0357985 , ...,  0.0258254 ,
        -0.01760606,  0.03873354],
       [ 0.04992054,  0.03843062,  0.01581663, ...,  0.03056639,
        -0.01087088,  0.02564374],
       ...,
       [ 0.01250061,  0.06084283,  0.09827578, ...,  0.02379247,
        -0.03498912,  0.02727959],
       [ 0.03516708, -0.00389013,  0.11047324, ...,  0.03990864,
        -0.02585147,  0.00556062],
       [ 0.07118661, -0.05717744,  0.02033144, ...,  0.03446807,
        -0.02102342,  0.0098958 ]], dtype=float32)

In [10]:
# embds = np.load("data/inter/meal_names.npz.npy")
meal_names['embedding'] = outputs.tolist()
meal_names.head()

,meal_id,meal,embedding
4,7010,Aurajuusto-pinaattilasagnette,"[0.011745614930987358, -0.038602955639362335, ..."
5,7010,Aurajuusto-pinaattilasagnettea,"[0.01333621982485056, -0.04670437052845955, 0...."
9,1751,BBQ-broilerikastiketta,"[0.049920544028282166, 0.03843062371015549, 0...."
10,1751,BBQ-Broilerikastiketta,"[0.0653582438826561, 0.038124874234199524, 0.0..."
13,200006,Bar Myöhä Bbq-seitanbowl,"[0.02153756469488144, 0.027581898495554924, 0...."


In [11]:
path = "data/inter/meal_names_embds.parquet"
meal_names.to_parquet(path)

# Calculate the cosine similari